# 01: Quickstart - Analyzing Your First World Model

This notebook demonstrates the core World Model Lens API for analyzing any world model.

In [ ]:
# Install dependencies
!pip install world_model_lens torch numpy

In [ ]:
from world_model_lens import HookedWorldModel, WorldModelConfig
from world_model_lens.backends.generic_adapter import WorldModelAdapter
import torch

## 1. Create a Simple World Model Adapter

World Model Lens works with ANY world model. Let's create a simple RSSM-like model.

In [ ]:
class SimpleRSSM(WorldModelAdapter):
    def __init__(self):
        super().__init__(None)
        self.encoder = torch.nn.Linear(128, 256)
        self.gru = torch.nn.GRUCell(256 + 4, 256)
        self.posterior_net = torch.nn.Linear(256, 32)
        self.prior_net = torch.nn.Linear(256, 32)
        self.reward_head = torch.nn.Linear(256, 1)
        self.decoder = torch.nn.Linear(32, 128)
        
    def encode(self, obs, context=None):
        obs_enc = self.encoder(obs)
        return obs_enc, obs_enc
    
    def dynamics(self, state, action=None):
        if action is not None:
            gru_input = torch.cat([state.hidden, action], dim=-1)
        else:
            gru_input = state.hidden
        new_hidden = self.gru(gru_input)
        return new_hidden
    
    def get_components(self):
        return ['encoder', 'gru', 'posterior', 'prior', 'reward']

## 2. Wrap and Analyze

In [ ]:
# Create hooked world model
model = SimpleRSSM()
wm = HookedWorldModel(adapter=model, config=WorldModelConfig())

# Generate observations and actions
obs_seq = torch.randn(20, 128)  # 20 timesteps
action_seq = torch.randn(20, 4)   # 20 timesteps

# Run with activation cache
traj, cache = wm.run_with_cache(obs_seq, action_seq)
print(f"Trajectory length: {len(traj)}")
print(f"Cache keys: {len(list(cache.keys()))} activations")

## 3. Analyze Latent Dynamics

In [ ]:
# Compute surprise (KL between posterior and prior)
surprise = cache.surprise()
print(f"Mean surprise: {surprise.mean():.4f}")

# Find surprise peaks
peaks = [(i, v) for i, v in enumerate(surprise.tolist()) if v > surprise.mean() + surprise.std()]
print(f"Surprise peaks at timesteps: {[p[0] for p in peaks]}")

## 4. Imagine Future Trajectories

In [ ]:
# Fork imagination from last state
start_state = traj.states[-1]
imagined = wm.imagine(start_state, horizon=10)
print(f"Imagined {len(imagined)} future states")

## 5. Memory Estimation

World Model Lens supports lazy evaluation for large models.

In [ ]:
# Estimate memory before materializing
mem_gb = cache.estimate_memory_gb()
print(f"Estimated cache size: {mem_gb:.4f} GB")

# Materialize only specific components
cache.materialize(names=['gru'], timesteps=[0, 5, 10])
mem_after = cache.estimate_memory_gb()
print(f"After partial materialize: {mem_after:.4f} GB")

## Next Steps

- **02_patching_basics**: Learn activation patching
- **03_causal_tracing**: Discover causal pathways
- **04_sparse_autoencoders**: Feature discovery with SAEs